In [ ]:
!pip install -U tensorflow_datasets tensorflow
!pip install -U importlib_resources


In [ ]:



import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import gc

from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)

In [ ]:
SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

IMG_SIZE = 224
NUM_CLASSES = 37
BATCH_SIZE = 32
EPOCHS = 5
AUTOTUNE = tf.data.AUTOTUNE

# Increase these values if GPU memory/runtime permits
TRAIN_SIZE = 2500
VAL_SIZE = 500
TEST_SIZE = 500


In [ ]:
(train_full, test_full), info = tfds.load(
    "oxford_iiit_pet",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)

print(info)


In [ ]:
train_raw = train_full.take(TRAIN_SIZE)
val_raw = test_full.take(VAL_SIZE)
final_test_raw = test_full.skip(VAL_SIZE).take(TEST_SIZE)

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, label

def create_dataset(dataset, batch_size=BATCH_SIZE, shuffle=False):
    if shuffle:
        dataset = dataset.shuffle(1000, seed=SEED)

    return (
        dataset
        .map(preprocess, num_parallel_calls=AUTOTUNE)
        .batch(batch_size)
        .prefetch(AUTOTUNE)
    )

train_ds = create_dataset(train_raw, shuffle=True)
val_ds = create_dataset(val_raw)
test_ds = create_dataset(final_test_raw)


In [ ]:
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

In [ ]:
from google.colab import files
def save_plot(filename):

    path = f"{filename}.pdf"
    plt.savefig(path, format='pdf', dpi=600, bbox_inches='tight')
    files.download(path)
    print(f"Saved and downloaded: {path}")

In [ ]:
plt.figure(figsize=(10, 8))

for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        image = (images[i].numpy() + 1) / 2
        plt.imshow(image)
        plt.title(f"Class: {labels[i].numpy()}")
        plt.axis("off")

plt.tight_layout()
plt.show()


In [2]:
def create_model(
    dropout_rate=0.0,
    l2_reg=0.0,
    batch_norm=False,
    initializer="glorot_uniform",
    trainable_base=False,
    optimizer="adam",
    learning_rate=0.001
):
    base_model = MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    base_model.trainable = trainable_base

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)

    if batch_norm:
        x = layers.BatchNormalization()(x)

    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        kernel_initializer=initializer,
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)

    model = tf.keras.Model(inputs, outputs)

    if optimizer == "sgd":
        opt = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer == "momentum":
        opt = tf.keras.optimizers.SGD(
            learning_rate=learning_rate,
            momentum=0.9
        )
    elif optimizer == "rmsprop":
        opt = tf.keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=opt,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [ ]:
import tensorflow as tf

initializers = {
    "Zero": "zeros",
    "Random": tf.keras.initializers.RandomNormal(stddev=0.05),
    "Xavier": "glorot_uniform",
    "He": "he_normal"
}

init_histories = {}

for name, initializer in initializers.items():
    print(f"Training with {name} initialization")

    tf.keras.backend.clear_session()

    model = create_model(
        initializer=initializer,
        optimizer="adam",
        learning_rate=0.001
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        verbose=1
    )

    init_histories[name] = history.history

    del model
    gc.collect()

In [4]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(8, 5))

for name, history in init_histories.items():
    plt.plot(history["loss"], label=name)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss vs Epoch for Different Initializations")
plt.legend()
plt.grid()
save_plot("loss_epoch")
plt.show()

NameError: name 'plt' is not defined

In [ ]:
plt.figure(figsize=(8, 5))

for name, history in init_histories.items():
    plt.plot(
        np.array(history["val_accuracy"]) * 100,
        label=name
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.title("Validation Accuracy vs Epoch for Different Initializations")
plt.legend()
plt.grid()
save_plot("accuracy_epoch")
plt.show()


In [5]:
import tensorflow as tf
import gc

regularization_configs = {
    "No Regularization": {
        "dropout_rate": 0.0,
        "l2_reg": 0.0,
        "batch_norm": False
    },
    "L2": {
        "dropout_rate": 0.0,
        "l2_reg": 0.001,
        "batch_norm": False
    },
    "Dropout": {
        "dropout_rate": 0.5,
        "l2_reg": 0.0,
        "batch_norm": False
    },
    "BatchNorm": {
        "dropout_rate": 0.0,
        "l2_reg": 0.0,
        "batch_norm": True
    }
}

reg_histories = {}

for name, config in regularization_configs.items():
    print(f"Training: {name}")

    tf.keras.backend.clear_session()

    model = create_model(**config)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        verbose=1
    )

    reg_histories[name] = history.history

    del model
    gc.collect()

Training: No Regularization


NameError: name 'tf' is not defined

In [ ]:
for name, history in reg_histories.items():
    plt.figure(figsize=(7, 4))

    plt.plot(np.array(history["accuracy"]) * 100, label="Training")
    plt.plot(np.array(history["val_accuracy"]) * 100, label="Validation")

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(f"Training vs Validation Accuracy – {name}")
    plt.legend()
    plt.grid()
    save_plot("reg")
    plt.show()


In [ ]:
for name, history in reg_histories.items():
    plt.figure(figsize=(7, 4))

    plt.plot(history["loss"], label="Training")
    plt.plot(history["val_loss"], label="Validation")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training vs Validation Loss – {name}")
    plt.legend()
    plt.grid()
    save_plot("no_reg")
    plt.show()


In [ ]:
bn_histories = {}

for use_bn in [False, True]:
    name = "With BatchNorm" if use_bn else "Without BatchNorm"

    tf.keras.backend.clear_session()

    model = create_model(batch_norm=use_bn)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        verbose=1
    )

    bn_histories[name] = history.history

    del model
    gc.collect()


In [ ]:
plt.figure(figsize=(8, 5))

for name, history in bn_histories.items():
    plt.plot(
        np.array(history["val_accuracy"]) * 100,
        label=name
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.title("With vs Without Batch Normalization")
plt.legend()
plt.grid()
save_plot("batch_norm")
plt.show()


In [6]:
import tensorflow as tf
import numpy as np
import pandas as pd
import time
import gc

optimizers = ["sgd", "momentum", "rmsprop", "adam"]

optimizer_histories = {}
optimizer_results = []

for opt_name in optimizers:
    print(f"Training with {opt_name}")

    tf.keras.backend.clear_session()

    model = create_model(
        optimizer=opt_name,
        learning_rate=0.001
    )

    start_time = time.time()

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        verbose=1
    )

    elapsed_time = time.time() - start_time

    optimizer_histories[opt_name] = history.history

    # Calculate convergence epoch
    if history.history["val_accuracy"]:
        convergence_epoch = np.argmax(history.history["val_accuracy"]) + 1
    else:
        convergence_epoch = np.nan

    optimizer_results.append({
        "Optimizer": opt_name,
        "Final Loss": history.history["loss"][-1],
        "Best Validation Accuracy (%)": max(history.history["val_accuracy"]) * 100,
        "Convergence Epoch": convergence_epoch,
        "Training Time (s)": elapsed_time
    })

    del model
    gc.collect()

optimizer_df = pd.DataFrame(optimizer_results)
display(optimizer_df)

Training with sgd


NameError: name 'tf' is not defined

In [ ]:
plt.figure(figsize=(8, 5))

for name, history in optimizer_histories.items():
    plt.plot(history["loss"], label=name.upper())

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss vs Epoch for Different Optimizers")
plt.legend()
plt.grid()
save_plot("optimizers")
plt.show()


In [7]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(8, 5))

for name, history in optimizer_histories.items():
    plt.plot(
        np.array(history["val_accuracy"]) * 100,
        label=name.upper()
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.title("Validation Accuracy vs Epoch for Different Optimizers")
plt.legend()
plt.grid()
save_plot("acc_optimizers")
plt.show()

NameError: name 'plt' is not defined

In [ ]:
learning_rates = [0.001, 0.0001]
lr_results = []

for lr in learning_rates:
    print(f"Learning rate: {lr}")

    tf.keras.backend.clear_session()

    model = create_model(
        optimizer="adam",
        learning_rate=lr
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        verbose=0
    )

    lr_results.append({
        "Learning Rate": lr,
        "Best Validation Accuracy (%)": max(history.history["val_accuracy"]) * 100
    })

    del model
    gc.collect()

lr_df = pd.DataFrame(lr_results)
lr_df


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(lr_df["Learning Rate"].astype(str), lr_df["Best Validation Accuracy (%)"])
plt.xlabel("Learning Rate")
plt.ylabel("Validation Accuracy (%)")
plt.title("Learning Rate vs Validation Accuracy")
save_plot("lr_val_acc")
plt.show()


In [ ]:
batch_sizes = [16, 32, 64]
batch_results = []

for batch_size in batch_sizes:
    print(f"Batch size: {batch_size}")

    train_temp = create_dataset(train_raw, batch_size=batch_size, shuffle=True)
    val_temp = create_dataset(val_raw, batch_size=batch_size)

    tf.keras.backend.clear_session()

    model = create_model()

    history = model.fit(
        train_temp,
        validation_data=val_temp,
        epochs=EPOCHS,
        verbose=0
    )

    batch_results.append({
        "Batch Size": batch_size,
        "Best Validation Accuracy (%)": max(history.history["val_accuracy"]) * 100
    })

    del model
    gc.collect()

batch_df = pd.DataFrame(batch_results)
batch_df


In [8]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.bar(batch_df["Batch Size"].astype(str), batch_df["Best Validation Accuracy (%)"])
plt.xlabel("Batch Size")
plt.ylabel("Validation Accuracy (%)")
plt.title("Batch Size vs Validation Accuracy")
save_plot("bsize_val_acc")
plt.show()

NameError: name 'plt' is not defined

In [ ]:
dropout_rates = [0.0, 0.25, 0.5]
dropout_results = []

for dropout in dropout_rates:
    print(f"Dropout rate: {dropout}")

    tf.keras.backend.clear_session()

    model = create_model(dropout_rate=dropout)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        verbose=0
    )

    dropout_results.append({
        "Dropout Rate": dropout,
        "Best Validation Accuracy (%)": max(history.history["val_accuracy"]) * 100
    })

    del model
    gc.collect()

dropout_df = pd.DataFrame(dropout_results)
dropout_df


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(dropout_df["Dropout Rate"].astype(str), dropout_df["Best Validation Accuracy (%)"])
plt.xlabel("Dropout Rate")
plt.ylabel("Validation Accuracy (%)")
plt.title("Dropout Rate vs Validation Accuracy")
save_plot("dropout_val_acc")
plt.show()


In [ ]:
tf.keras.backend.clear_session()

feature_model = create_model(
    dropout_rate=0.25,
    optimizer="adam",
    learning_rate=0.001,
    trainable_base=False
)

feature_history = feature_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1
)


In [ ]:
base_model = None

for layer in feature_model.layers:
    if isinstance(layer, tf.keras.Model):
        base_model = layer
        break

base_model.trainable = True

FINE_TUNE_AT = 100

for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

feature_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fine_tune_history = feature_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1
)


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    np.array(feature_history.history["val_accuracy"]) * 100,
    label="Feature Extraction"
)

plt.plot(
    np.array(fine_tune_history.history["val_accuracy"]) * 100,
    label="Fine-Tuning"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.title("Feature Extraction vs Fine-Tuning")
plt.legend()
plt.grid()
save_plot("feature_vs_fine_tune")
plt.show()


In [1]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(feature_history.history["val_loss"], label="Feature Extraction")
plt.plot(fine_tune_history.history["val_loss"], label="Fine-Tuning")

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss Before and After Fine-Tuning")
plt.legend()
plt.grid()
save_plot("val_loss_finetune")
plt.show()

NameError: name 'plt' is not defined

In [ ]:
CV_SAMPLES = 1000

X_list = []
y_list = []

for image, label in train_raw.take(CV_SAMPLES):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)

    X_list.append(image.numpy())
    y_list.append(label.numpy())

X = np.array(X_list)
y = np.array(y_list)

print("X shape:", X.shape)
print("y shape:", y.shape)


In [9]:
configs = {
    "C1": {
        "dropout_rate": 0.0,
        "optimizer": "adam",
        "learning_rate": 0.001
    },
    "C2": {
        "dropout_rate": 0.25,
        "optimizer": "adam",
        "learning_rate": 0.001
    },
    "C3": {
        "dropout_rate": 0.5,
        "optimizer": "adam",
        "learning_rate": 0.0001
    }
}


In [10]:
from sklearn.model_selection import KFold
import tensorflow as tf
import numpy as np
import gc

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

cv_results = {}

for config_name, config in configs.items():
    print(f"Running cross-validation for {config_name}")

    fold_scores = []

    for fold, (train_index, val_index) in enumerate(kf.split(X)):
        print(f"Fold {fold + 1}/5")

        X_train = X[train_index]
        X_val = X[val_index]

        y_train = y[train_index]
        y_val = y[val_index]

        tf.keras.backend.clear_session()

        model = create_model(
            dropout_rate=config["dropout_rate"],
            optimizer=config["optimizer"],
            learning_rate=config["learning_rate"]
        )

        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=3,
            batch_size=32,
            verbose=0
        )

        fold_scores.append(
            max(history.history["val_accuracy"])
        )

        del model
        gc.collect()

    cv_results[config_name] = fold_scores

NameError: name 'KFold' is not defined

In [11]:
import numpy as np
import pandas as pd

cv_table = []

for config_name, scores in cv_results.items():
    cv_table.append({
        "Configuration": config_name,
        "F1": scores[0] * 100,
        "F2": scores[1] * 100,
        "F3": scores[2] * 100,
        "F4": scores[3] * 100,
        "F5": scores[4] * 100,
        "Mean Accuracy (%)": np.mean(scores) * 100,
        "SD (%)": np.std(scores) * 100
    })

cv_df = pd.DataFrame(cv_table)
cv_df

NameError: name 'cv_results' is not defined

In [ ]:
plt.figure(figsize=(8, 5))

plt.errorbar(
    cv_df["Configuration"],
    cv_df["Mean Accuracy (%)"],
    yerr=cv_df["SD (%)"],
    fmt="o",
    capsize=5
)

plt.xlabel("Hyperparameter Configuration")
plt.ylabel("Mean Validation Accuracy (%)")
plt.title("5-Fold Cross-Validation Accuracy")
plt.grid()
save_plot("cv_acc")
plt.show()


In [ ]:
best_config_name = cv_df.loc[
    cv_df["Mean Accuracy (%)"].idxmax(),
    "Configuration"
]

best_config = configs[best_config_name]

print("Best configuration:", best_config_name)
print(best_config)


In [ ]:
tf.keras.backend.clear_session()

final_model = create_model(
    dropout_rate=best_config["dropout_rate"],
    optimizer=best_config["optimizer"],
    learning_rate=best_config["learning_rate"]
)

start_time = time.time()

final_history = final_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1
)

training_time = time.time() - start_time

print(f"Training time: {training_time:.2f} seconds")


In [ ]:
test_loss, test_accuracy = final_model.evaluate(
    test_ds,
    verbose=1
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


In [ ]:
y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = final_model.predict(images, verbose=0)
    predicted_labels = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall   : {recall * 100:.2f}%")
print(f"F1 Score : {f1 * 100:.2f}%")


In [ ]:
print("Number of parameters:", final_model.count_params())


In [ ]:
best_cv_row = cv_df[
    cv_df["Configuration"] == best_config_name
].iloc[0]

final_results = pd.DataFrame({
    "Metric": [
        "Mean CV Accuracy",
        "CV Standard Deviation",
        "Test Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Training Time (seconds)",
        "Number of Parameters"
    ],
    "Value": [
        f"{best_cv_row['Mean Accuracy (%)']:.2f}%",
        f"{best_cv_row['SD (%)']:.2f}%",
        f"{accuracy * 100:.2f}%",
        f"{precision * 100:.2f}%",
        f"{recall * 100:.2f}%",
        f"{f1 * 100:.2f}%",
        f"{training_time:.2f}",
        final_model.count_params()
    ]
})

final_results


In [ ]:
# Confusion Matrix - Annotated Heatmap

import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

# Get class names from the Oxford-IIIT Pet dataset
class_names = info.features["label"].names

plt.figure(figsize=(18, 16))

sns.heatmap(
    cm,
    annot=True,          # Show numbers in each cell
    fmt="d",             # Integer format
    cmap="Blues",
    cbar=True,
    xticklabels=class_names,
    yticklabels=class_names,
    annot_kws={"size": 8}
)

plt.title(
    "MobileNetV2 Confusion Matrix",
    fontsize=18,
    pad=20
)

plt.xlabel(
    "Predicted Label",
    fontsize=14
)

plt.ylabel(
    "True Label",
    fontsize=14
)

plt.xticks(
    rotation=90,
    fontsize=9
)

plt.yticks(
    rotation=0,
    fontsize=9
)

plt.tight_layout()
save_plot("confusion_matrix")
plt.show()

In [12]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import pandas as pd
import time, gc

from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

# ---- same setup as your notebook ----
SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

IMG_SIZE = 224
NUM_CLASSES = 37
BATCH_SIZE = 32
EPOCHS = 5
AUTOTUNE = tf.data.AUTOTUNE

TRAIN_SIZE = 2500
VAL_SIZE = 500
TEST_SIZE = 500

(train_full, test_full), info = tfds.load(
    "oxford_iiit_pet", split=["train", "test"], as_supervised=True, with_info=True
)

train_raw = train_full.take(TRAIN_SIZE)
val_raw = test_full.take(VAL_SIZE)
final_test_raw = test_full.skip(VAL_SIZE).take(TEST_SIZE)

In [13]:
def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, label

def create_dataset(dataset, batch_size=BATCH_SIZE, shuffle=False):
    if shuffle:
        dataset = dataset.shuffle(1000, seed=SEED)
    return (dataset.map(preprocess, num_parallel_calls=AUTOTUNE)
                    .batch(batch_size).prefetch(AUTOTUNE))

train_ds = create_dataset(train_raw, shuffle=True)
val_ds = create_dataset(val_raw)
test_ds = create_dataset(final_test_raw)

def create_model(dropout_rate=0.0, l2_reg=0.0, batch_norm=False,
                  initializer="glorot_uniform", trainable_base=False,
                  optimizer="adam", learning_rate=0.001):
    base_model = MobileNetV2(include_top=False, weights="imagenet",
                              input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base_model.trainable = trainable_base
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    if batch_norm:
        x = layers.BatchNormalization()(x)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax",
                            kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(l2_reg))(x)
    model = tf.keras.Model(inputs, outputs)
    if optimizer == "sgd":
        opt = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer == "momentum":
        opt = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    elif optimizer == "rmsprop":
        opt = tf.keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=opt, loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

In [14]:
# ---- rebuild CV sample set exactly like cell 30 ----
CV_SAMPLES = 1000
X_list, y_list = [], []
for image, label in train_raw.take(CV_SAMPLES):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    X_list.append(image.numpy())
    y_list.append(label.numpy())
X = np.array(X_list)
y = np.array(y_list)

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def run_cv(model_fn):
    """model_fn() -> compiled model. Returns fold scores list."""
    fold_scores = []
    for train_index, val_index in kf.split(X):
        X_train, X_val = X[train_index], X[val_index]
        y_train, y_val = y[train_index], y[val_index]
        tf.keras.backend.clear_session()
        model = model_fn()
        history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                             epochs=3, batch_size=32, verbose=0)
        fold_scores.append(max(history.history["val_accuracy"]))
        del model
        gc.collect()
    return fold_scores

def evaluate_on_test(model, ds):
    loss, acc = model.evaluate(ds, verbose=0)
    return acc


In [ ]:

results = {}

# ---------------- Best Initialization: Zero init ----------------
print("=== Best Initialization (Zero) ===")
scores = run_cv(lambda: create_model(initializer="zeros", optimizer="adam", learning_rate=0.001))
tf.keras.backend.clear_session()
model = create_model(initializer="zeros", optimizer="adam", learning_rate=0.001)
t0 = time.time()
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=0)
train_time = time.time() - t0
test_acc = evaluate_on_test(model, test_ds)
results["Best Initialization"] = {
    "CV Accuracy": np.mean(scores) * 100,
    "SD": np.std(scores) * 100,
    "Test Accuracy": test_acc * 100,
    "Training Time": train_time
}
del model; gc.collect()

=== Best Initialization (Zero) ===


In [ ]:

print("=== Best Optimizer (RMSprop) ===")
scores = run_cv(lambda: create_model(optimizer="rmsprop", learning_rate=0.001))
tf.keras.backend.clear_session()
model = create_model(optimizer="rmsprop", learning_rate=0.001)
t0 = time.time()
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=0)
train_time = time.time() - t0
test_acc = evaluate_on_test(model, test_ds)
results["Best Optimizer"] = {
    "CV Accuracy": np.mean(scores) * 100,
    "SD": np.std(scores) * 100,
    "Test Accuracy": test_acc * 100,
    "Training Time": train_time
}
del model; gc.collect()

In [ ]:

print("=== Fine-Tuned Model ===")
def build_and_finetune():
    m = create_model(dropout_rate=0.25, optimizer="adam", learning_rate=0.001,
                      trainable_base=False)
    return m

def finetune_cv_fold(X_train, y_train, X_val, y_val):
    tf.keras.backend.clear_session()
    model = build_and_finetune()
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=2, batch_size=32, verbose=0)
    base_model = None
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            base_model = layer
            break
    base_model.trainable = True
    FINE_TUNE_AT = 100
    for layer in base_model.layers[:FINE_TUNE_AT]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                         epochs=2, batch_size=32, verbose=0)
    acc = max(history.history["val_accuracy"])
    del model
    gc.collect()
    return acc

fold_scores = []
for train_index, val_index in kf.split(X):
    acc = finetune_cv_fold(X[train_index], y[train_index], X[val_index], y[val_index])
    fold_scores.append(acc)


In [ ]:

t0 = time.time()
tf.keras.backend.clear_session()
ft_model = build_and_finetune()
ft_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=0)
base_model = None
for layer in ft_model.layers:
    if isinstance(layer, tf.keras.Model):
        base_model = layer
        break
base_model.trainable = True
FINE_TUNE_AT = 100
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False
ft_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
ft_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=0)
train_time = time.time() - t0
test_acc = evaluate_on_test(ft_model, test_ds)

results["Fine-Tuned Model"] = {
    "CV Accuracy": np.mean(fold_scores) * 100,
    "SD": np.std(fold_scores) * 100,
    "Test Accuracy": test_acc * 100,
    "Training Time": train_time
}
del ft_model; gc.collect()

In [ ]:


reused = {
    "CV Accuracy": 85.90,
    "SD": 1.62,
    "Test Accuracy": 89.00,
    "Training Time": 670.86
}
results["Baseline"] = reused
results["Best Regularization"] = reused
results["Best Hyperparameters"] = reused

# ---------------- Final table ----------------
order = ["Baseline", "Best Initialization", "Best Regularization",
         "Best Optimizer", "Best Hyperparameters", "Fine-Tuned Model"]

table = pd.DataFrame([
    {"Configuration": name,
     "CV Accuracy": f"{results[name]['CV Accuracy']:.2f}%",
     "SD": f"{results[name]['SD']:.2f}%",
     "Test Accuracy": f"{results[name]['Test Accuracy']:.2f}%",
     "Training Time": f"{results[name]['Training Time']:.2f}s"}
    for name in order
])

table